# Navigation Model — Sensor Input → Movement Decision
**Robot:** SCITOS G5 · **Task:** Wall-following · **Sensors:** 24 ultrasonic sensors arranged 360° around the robot's waist

The robot reads distance values from every direction and must decide **what to do next**:

| Label | Meaning |
|---|---|
| `Move-Forward` | Clear path ahead |
| `Slight-Right-Turn` | Gentle right correction |
| `Sharp-Right-Turn` | Hard right needed |
| `Slight-Left-Turn` | Gentle left correction |

We train **four classifiers** and pick the best one.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.05)
PALETTE = ['#4C72B0','#55A868','#C44E52','#8172B2']
print('Libraries loaded ✔')

## 1 · Load Dataset

In [ ]:
col_names = [f'US{i}' for i in range(1, 25)] + ['class']

df = pd.read_csv(
    '../robot-intelligence-dataset/navigation_dataset/sensor_readings_24.data',
    header=None, names=col_names
)

print(f'Rows: {df.shape[0]}  |  Features: {df.shape[1]-1}  |  Missing: {df.isnull().sum().sum()}')
df.head(3)

## 2 · Exploratory Data Analysis

In [ ]:
# 2a  Class distribution
counts = df['class'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(counts.index, counts.values, color=PALETTE, edgecolor='white', linewidth=1.2)
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Movement Decision')
axes[0].set_ylabel('Sample Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Class Share')

plt.suptitle('Target Variable Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2b  Mean sensor readings per class
sensor_cols = [c for c in df.columns if c != 'class']
class_means = df.groupby('class')[sensor_cols].mean()

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(sensor_cols))
width = 0.2

for i, (cls, color) in enumerate(zip(class_means.index, PALETTE)):
    ax.bar(x + i * width, class_means.loc[cls], width, label=cls, color=color, alpha=0.87)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(sensor_cols, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Mean Distance (m)')
ax.set_title('Average Sensor Reading per Class — all 24 ultrasonic sensors')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 2c  Sensor correlation heatmap (lower triangle)
corr = df[sensor_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Sensor Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 3 · Preprocess

In [ ]:
X = df[sensor_cols].values
y_raw = df['class'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
print('Encoded classes:', dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()          # critical for SVM and KNN
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape[0]}  |  Test: {X_test_sc.shape[0]}')

## 4 · Train Models

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM'          : SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    'KNN'          : KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
}

results = {}
for name, mdl in models.items():
    mdl.fit(X_train_sc, y_train)
    preds  = mdl.predict(X_test_sc)
    acc    = accuracy_score(y_test, preds)
    cv_acc = cross_val_score(mdl, X_train_sc, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    results[name] = {
        'model'   : mdl,
        'preds'   : preds,
        'accuracy': acc,
        'cv_mean' : cv_acc.mean(),
        'cv_std'  : cv_acc.std(),
        'cm'      : confusion_matrix(y_test, preds),
    }
    print(f'{name:15s}  test={acc:.4f}  cv={cv_acc.mean():.4f} ± {cv_acc.std():.4f}')

## 5 · Evaluate — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()
labels = le.classes_

for ax, (name, r) in zip(axes, results.items()):
    disp = ConfusionMatrixDisplay(r['cm'], display_labels=labels)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=30)
    ax.set_title(f'{name}\nTest Accuracy: {r["accuracy"]:.4f}', fontsize=11, pad=10)

plt.suptitle('Confusion Matrices — Navigation Task', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6 · Compare Models

In [ ]:
summary = pd.DataFrame([
    {'Model': k, 'Test Accuracy': v['accuracy'],
     'CV Mean': v['cv_mean'], 'CV Std': v['cv_std']}
    for k, v in results.items()
]).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)

print(summary.to_string(index=False, float_format='{:.4f}'.format))

best_name = summary.iloc[0]['Model']
print(f'\n🏆  Best model: {best_name} ({summary.iloc[0]["Test Accuracy"]:.4f})')

In [ ]:
names = summary['Model'].tolist()
x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, summary['Test Accuracy'], w, label='Test Accuracy',    color=PALETTE[0], alpha=0.9)
b2 = ax.bar(x + w/2, summary['CV Mean'],       w, label='CV Mean (5-fold)', color=PALETTE[1], alpha=0.9)
ax.errorbar(x + w/2, summary['CV Mean'], yerr=summary['CV Std'],
            fmt='none', c='black', capsize=5, linewidth=1.5)

for bar in [*b1, *b2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0.7, 1.04)
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison: Test vs 5-fold CV Accuracy', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 7 · Feature Importance (Random Forest)

In [ ]:
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=sensor_cols).sort_values(ascending=True)
colors = ['#C44E52' if imp > importances.median() else '#4C72B0' for imp in importances]

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(importances.index, importances.values, color=colors)
ax.axvline(importances.median(), color='orange', linestyle='--', linewidth=1.5, label='Median')
ax.set_xlabel('Feature Importance')
ax.set_title('Sensor Importance — Random Forest', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print('Top 5 most informative sensors:')
for s, v in importances.sort_values(ascending=False).head(5).items():
    print(f'  {s}: {v:.4f}')

## 8 · Best Model — Full Classification Report

In [ ]:
best_res = results[best_name]
print(f'🏆 Best model : {best_name}')
print(f'   Test Accuracy : {best_res["accuracy"]:.4f}')
print(f'   CV Accuracy   : {best_res["cv_mean"]:.4f} ± {best_res["cv_std"]:.4f}')
print()
print(classification_report(y_test, best_res['preds'], target_names=labels))